In [87]:
from sklearn import preprocessing
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score, mean_absolute_percentage_error


# For 2D analysis
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from scipy.optimize import curve_fit
from sklearn.preprocessing import MinMaxScaler
# from utils import period2freq, freq2period

# For PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import csv
import time
import glob
import os

# Datasize in KB
data_size_kb = {'4mb': 4096, '16mb': 16384, '64mb': 65536,
            '256mb': 262144, '512mb': 524288, '1gb': 1048576,
            '5gb': 5242880, '50gb': 52428800, '100gb': 104857600,
            '300gb': 314572800,}

# Key Parameters
IOR_PARAMS = ['operation', 'randomOffset', 'transferSize', 
            'aggregateFilesizeMB', 'numTasks', 'totalTime', 
            'numNodes', 'tasksPerNode', 'bwMiB', "storageType"]

TARGET_PARAMS = [ "bestStorage" ]
op_dict = {0: "write", 1: "read"}


# Parameter Notes for Datalife:
Each entry in the table represent only one single edge in the workflow. An directed edge connects a **fileName** and a **taskName**, representing data access.

---
- **operation**: The type of I/O operation {0: "write", 1: "read"}, value 1 represents read (e.g. a directed edge edge from a **fileName** to a **taskName**), value 0 represents write (e.g. a directed edge edge from a **taskName** to a **fileName**)
- **randomOffset**: The type of data access pattern { 0: "sequential file access", 1: "random file access"}
- **transferSize**: Average I/O size of the particular I/O operation to a file, calculated from aggregateFilesizeMB/opCount
- **aggregateFilesizeMB**: Total I/O size of a particular I/O operation to a file for a task
- **numTasks**: Number of parallel tasks for this particular task
- **totalTime**: The total I/O time of of a particular I/O operation to a file for a task
- **numNodes**: Number of nodes used for this particular task
- **tasksPerNode**: numTasks/numNodes for a task
- **bwMiB**: bandwidth of a particular I/O operation to a file for a task, calculated from aggregateFilesizeMB/totalTime
- **storageType**: The storage type used in this task. {0: "localssd", 1: "beegfs/pfs", 2: "lustre", 3: "unknown"}
- **opCount**: the number of I/O operation count of a particular I/O operation to a file for a task
- **taskName**: the task name that is running for a particular workflow
- **taskPID**: the task PID
- **fileName**: the name of file that a I/O operation is for

In [88]:
def byte_size_to_human_size(byte_size):
    if byte_size < 1024:
        return f"{byte_size} B"
    elif byte_size < 1024**2:
        return f"{byte_size/1024} KiB"
    elif byte_size < 1024**3:
        return f"{byte_size/1024**2} MiB"
    elif byte_size < 1024**4:
        return f"{byte_size/1024**3} GiB"
    else:
        return f"{byte_size/1024**4} TiB"


def file_size_to_mb(file_size):
    # If file_size is a string, convert to float
    if isinstance(file_size, str):
        # Translate KiB, MiB, and GiB to bytes
        size_num, size_unit = file_size.split()
        size_num = float(size_num)
        
        if size_unit == "KiB":
            return size_num / 1024  # Convert KiB to MB
        elif size_unit == "MiB":
            return size_num  # Already in MB
        elif size_unit == "GiB":
            return size_num * 1024  # Convert GiB to MB
        else:
            raise ValueError(f"Unknown size unit: {size_unit}")
    elif isinstance(file_size, (int, float)):
        # If file_size is an integer or float, assume it's in bytes and convert to MB
        return file_size / (1024 ** 2)  # Convert bytes to MB
    else:
        raise TypeError("file_size must be a string or a number")

def transform_store_code(storage_type):
    if storage_type == "localssd":
        store_code = 0
    elif storage_type == "beegfs" or storage_type == "pfs":
        store_code = 1
    elif storage_type == "lustre":
        store_code = 2
    else:
        store_code = 3
    return store_code

def decode_store_code(store_code):
    if store_code == 0:
        storage_type = "localssd"
    elif store_code == 1:
        storage_type = "beegfs"
    elif store_code == 2:
        storage_type = "lustre"
    else:
        storage_type = "unknown"
    return storage_type

def get_ior_json_df(ior_test_json_files, storage_type, IOR_PARAMS):


    # Create a dataframe for storing ior parameters
    ior_df = pd.DataFrame(columns=IOR_PARAMS)
    store_code = transform_store_code(storage_type)

    # Load Json files
    for jfile in ior_test_json_files:
        with open(jfile) as f:
            try:
                data = json.load(f)
            except:
                print("Error reading file: ", jfile)
                continue
            # print(jfile)
            
            # Write then Read
            ior_options = data['tests'][0]["Options"]
            IOR_PARAMS = data['tests'][0]["Parameters"]

            # Add write statistics first
            if file_size_to_mb(ior_options['aggregate filesize']) != 0:
                tmp_write_stat = {}
                ior_write_summary = data['summary'][0]
                tmp_write_stat['randomOffset'] = IOR_PARAMS['randomOffset']
                tmp_write_stat['aggregateFilesizeMB'] = file_size_to_mb(ior_options['aggregate filesize'])
                tmp_write_stat['numTasks'] = ior_write_summary['numTasks']
                tmp_write_stat['tasksPerNode'] = ior_write_summary['tasksPerNode']
                tmp_write_stat['numNodes'] = tmp_write_stat['numTasks']/tmp_write_stat['tasksPerNode']
                tmp_write_stat['transferSize'] = ior_write_summary['transferSize']
                if ior_write_summary['operation'] == "write":
                    tmp_write_stat['operation'] = 0
                else:
                    tmp_write_stat['operation'] = -1
                tmp_write_stat['totalTime'] = ior_write_summary['MeanTime'] # only 1 operation per test
                tmp_write_stat['bwMiB'] = ior_write_summary['bwMeanMIB'] # only 1 operation per test
                tmp_write_stat['storageType'] = store_code
                # Add tmp_write_stat to df
                ior_df = ior_df._append(tmp_write_stat, ignore_index=True)

                # Add read statistics
                tmp_read_stat = {}
                ior_read_summary = data['summary'][1]
                tmp_read_stat['randomOffset'] = IOR_PARAMS['randomOffset']
                tmp_read_stat['aggregateFilesizeMB'] = file_size_to_mb(ior_options['aggregate filesize'])
                tmp_read_stat['numTasks'] = ior_read_summary['numTasks']
                tmp_read_stat['tasksPerNode'] = ior_read_summary['tasksPerNode']
                tmp_read_stat['numNodes'] = tmp_read_stat['numTasks']/tmp_read_stat['tasksPerNode']
                tmp_read_stat['transferSize'] = ior_read_summary['transferSize']
                if ior_read_summary['operation'] == "read":
                    tmp_read_stat['operation'] = 1
                else:
                    tmp_read_stat['operation'] = -1
                tmp_read_stat['totalTime'] = ior_read_summary['MeanTime']
                tmp_read_stat['bwMiB'] = ior_read_summary['bwMeanMIB']
                tmp_read_stat['storageType'] = store_code
                # Add tmp_read_stat to df
                ior_df = ior_df._append(tmp_read_stat, ignore_index=True)

    # encode string to numbers
    le = preprocessing.LabelEncoder()
    for col in ior_df.columns:
        if pd.api.types.is_string_dtype(ior_df[col]):
            ior_df[col] = le.fit_transform(ior_df[col])

    # print(ior_df.head(5))

    return ior_df

def plot_heatmap(corrM,outfile):
    plt.figure(figsize=(14, 8))
    #labels = list(corrM.columns)
    plt.subplots_adjust(bottom=0.19)

    # # use only lower triangle
    # corrM = corrM.where(np.triu(np.ones(corrM.shape)).astype(np.bool))

    ## plot all correlation heatmap
    map = sns.heatmap(corrM, vmin=-1, vmax=1,
        linewidths=0.5, linecolor='grey', cmap='BrBG') #annot=True,
    map.set_title('Correlation Matrix Heatmap', fontdict={'fontsize':12}, pad=12)
    #plt.show()
    out_file=f'{outfile}.png'
    plt.savefig(out_file)
    plt.clf()

def get_redundant_pairs(df):
    '''Get diagonal and lower triangular pairs of correlation matrix'''
    pairs_to_drop = set()
    cols = df.columns
    for i in range(0, df.shape[1]):
        for j in range(0, i+1):
            pairs_to_drop.add((cols[i], cols[j]))
    return pairs_to_drop
    
def corrM_sorted_csv(corrM, outfile):
  n=5
  au_corr = corrM.corr().abs().unstack()
  labels_to_drop = get_redundant_pairs(corrM)
  au_corr = au_corr.drop(labels=labels_to_drop).sort_values(ascending=False)
  sorted_corrM = au_corr[0:n]

  out_file= outfile
  sorted_corrM.to_csv(out_file)

def corr_matrix(df,outname=""):

    # calculte correlation matrix
    corrM = df.corr()

    corrM.to_csv(f'{outname}.csv')
    plot_heatmap(corrM,outname)
    #corrM_sorted_csv(corrM, f'sorted_{outname}.csv')

def _2d_trend(X, y, x_label="x-axis", y_label="bwMiB", title="", show=False):
        # X = X.reshape(-1, 1)
        X = X.values.reshape(-1, 1)
        # poly = PolynomialFeatures(degree=2)
        poly = PolynomialFeatures(degree=1)
        poly_data = poly.fit_transform(X)
        model = LinearRegression()
        model.fit(poly_data,y)
        coef = model.coef_
        intercept = model.intercept_
        # Set figure size (80,50)
        plt.figure(figsize=(10,5))

        plt.scatter(X,y,color='red')
        plt.plot(X,model.predict(poly.fit_transform(X)),color='blue')
        # Show the fit parameters on graph with scientific notation
        plt.annotate(f"y = {coef[1]:.2e}x + {intercept:.2e}", xy=(0.05, 0.95), xycoords='axes fraction')

        # print(f"y = {coef[1]}x + {intercept}")
        plt.legend(['Original','Prediction'])
        plt.xlabel(x_label)
        plt.ylabel(y_label)
        plt.title(title)
        if show:
            plt.show()
        return coef[1], intercept

def _3d_trend(x,y,z, title=""):
    # ref: https://stackoverflow.com/questions/2298390/fitting-a-line-in-3d
    # Convert Pandas Series to NumPy arrays
    x = x.to_numpy()
    y = y.to_numpy()
    z = z.to_numpy()
    
    data = np.concatenate((x[:, np.newaxis], 
                        y[:, np.newaxis], 
                        z[:, np.newaxis]), 
                        axis=1)
    
    data = data.astype('float64')

    # Calculate the mean of the points, i.e. the 'center' of the cloud
    datamean = data.mean(axis=0)
    print(datamean)

    # Do an SVD on the mean-centered data.
    # full_matrices=False reduce memory
    uu, dd, vv = np.linalg.svd(data - datamean, full_matrices=False)

    # Get the sptread of data with mean 0 from all axis
    x_min = np.min(data[:,0])
    y_min = np.min(data[:,1])
    z_min = np.min(data[:,2])

    x_max = np.max(data[:,0])
    y_max = np.max(data[:,1])
    z_max = np.max(data[:,2])

    low_bound = min(x_min, y_min, z_min)
    high_bound = max(x_max, y_max, z_max)

    # Now vv[0] contains the first principal component, i.e. the direction
    # vector of the 'best fit' line in the least squares sense.
    # Adjust axist limits (Optional)
    linepts = vv[0] * np.mgrid[low_bound:high_bound:2j][:, np.newaxis]
    # shift by the mean to get the line in the right place
    linepts += datamean

    # Verify that everything looks right.

    # import mpl_toolkits.mplot3d as m3d
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter3D(*data.T)
    ax.plot3D(*linepts.T)
    # ax.scatter3D(data[:,0], data[:,1], data[:,2])
    ax.set_xlabel("transferSize")
    ax.set_ylabel("aggregateFilesizeMB")
    ax.set_zlabel("bwMiB")
    ax.set_title(title)
    plt.show()



In [89]:
def ior_json_to_df(ior_data_path):
    # Get the list of JSON files from the directory
    ior_test_json_files = glob.glob(ior_data_path + "/*.json")

    # Get storage type from data_path name
    # storage_type =  ior_data_path.split("/")[-1].split("_")[0] + "_" + ior_data_path.split("/")[-1].split("_")[1]
    storage_type =  ior_data_path.split("/")[-1].split("_")[0]
    # print("Storage Type: ", storage_type)
    ior_df = get_ior_json_df(ior_test_json_files, storage_type, IOR_PARAMS)

    # corr_matrix(ior_df, storage_type)
    return ior_df

# Take a column name for y_data
def x_mean_std(group, y_column):
    return pd.Series({
        f'{y_column}_ave': group[y_column].mean(),
        f'{y_column}_ave_std': group[y_column].std(ddof=0)  # Use population standard deviation by setting ddof=0
    })


def my_data_transform(df, cols_to_norm=[], cols_to_log=[]):
    # Create an empty DataFrame to store the results
    results = []

    # Get unique transfer sizes and number of tasks
    xfersizes = sorted(df['transferSize'].unique())
    numTasks = sorted(df['numTasks'].unique())
    op_type = [0, 1]  # Define outside the loop to avoid redefinition
    print("Transfer Sizes: ", xfersizes)
    print("Number of Tasks: ", numTasks)

    for xfer in xfersizes:
        for t in numTasks:
            for op in op_type:
                # Filter the DataFrame for specific conditions
                subdf = df[(df['numTasks'] == t) &
                           (df['transferSize'] == xfer) &
                           (df['operation'] == op)].copy()
                
                if not subdf.empty:
                    # Calculate the mean and standard deviation of the bwMiB column
                    ave_bw_df = subdf.groupby('aggregateFilesizeMB').apply(
                        lambda group: x_mean_std(group, 'bwMiB')
                    ).reset_index()

                    # Calculate percentage of standard deviation
                    ave_bw_df['bwMiB_ave_std_perc'] = (ave_bw_df['bwMiB_ave_std'] /
                                                       ave_bw_df['bwMiB_ave']) * 100

                    # Merge the aggregated results with the original DataFrame
                    merged_df = pd.merge(subdf, ave_bw_df, on='aggregateFilesizeMB', how='left')

                    # Append the merged DataFrame to the results list
                    results.append(merged_df)

    # Concatenate all the DataFrames in the results list
    new_df = pd.concat(results, ignore_index=True)

    # Log transformation
    if cols_to_log:
        log_new_cols = [f"{col}_log" for col in cols_to_log]
        print("Columns to log: ", cols_to_log)
        new_df[log_new_cols] = np.log(new_df[cols_to_log])

    # Min-Max normalization
    if cols_to_norm:
        norm_new_cols = [f"{col}_norm" for col in cols_to_norm]
        print("Columns to normalize: ", cols_to_norm)
        scaler = MinMaxScaler()
        new_df[norm_new_cols] = scaler.fit_transform(new_df[cols_to_norm])

    return new_df

def oscillatory_func(x, amplitude, frequency, phase, offset):
    return amplitude * np.sin(frequency * x + phase) + offset

def damped_sine_func(x, amplitude, frequency, phase, offset, decay):
    return amplitude * np.sin(frequency * x + phase) * np.exp(-decay * x) + offset

def cos_func(x, amplitude, frequency):
    return amplitude * np.cos(frequency * x)

def normalize_y(y_data):
    scaler = MinMaxScaler(feature_range=(-1, 1))
    return scaler.fit_transform(y_data.values.reshape(-1, 1)).flatten()




In [90]:
def merge_by_test_param(ssd_df, beegfs_df):
    # Compare and find the row with only storage_type and bwMiB columns different but others are the same
    # Merge the dataframes on columns other than 'storageType' and 'bwMiB'
    merge_columns = ['operation', 'randomOffset', 'transferSize', 'aggregateFilesizeMB', 'numTasks', 'numNodes', 'tasksPerNode', 'opCount'] # 'totalTime', 
    merged_df = pd.merge(ssd_df, beegfs_df, on=merge_columns, suffixes=('_ssd', '_beegfs'), how='outer')

    # Remove columns storageType_ssd and storageType_beegfs
    merged_df.drop(['storageType_ssd', 'storageType_beegfs'], axis=1, inplace=True)
    
    # # Check if cloumens 'operation_ssd' and 'operation_beegfs' has the same values, if yes combine to 1 column 'operation'
    # merged_df['operation'] = merged_df.apply(lambda row: 0 if row['operation_ssd'] == row['operation_beegfs'] else -1, axis=1)

    # Compare 'bwMiB' values and add 'selectStorage' column
    merged_df['selectStorage'] = merged_df.apply(lambda row: 0 if row['bwMiB_ssd'] > row['bwMiB_beegfs'] else 1, axis=1)

    # Debug: Print unique values of 'operation' column before and after merge
    print("Unique 'operation' values in ssd_df:", ssd_df['operation'].unique())
    print("Unique 'operation' values in beegfs_df:", beegfs_df['operation'].unique())
    print("Unique 'operation' values in merged_df:", merged_df['operation'].unique())
    
    print(merged_df.head(5))
    # print shaoe
    print(f"merged_df.shape: {merged_df.shape}")
    print(f"ssd_df.shape: {ssd_df.shape}")
    print(f"beegfs_df.shape: {beegfs_df.shape}")

    return merged_df




In [91]:
def is_sequential(numbers):
    if not numbers:  # Check if the list is empty
        return False

    sorted_numbers = sorted(numbers)  # Sort the numbers
    return all(sorted_numbers[i] + 1 == sorted_numbers[i + 1] for i in range(len(sorted_numbers) - 1))


def get_stat_file_pids(all_files):
    # Extract target tasks from blk_files
    target_tasks = set()
    for blk_file in all_files:
        # Get the filename without the path
        filename = os.path.basename(blk_file)
        # repalce ".local" for now
        filename = filename.replace(".local", "")
        
        # Split filename by '.'
        parts = filename.split('.')
        # print(f"get_stat_file_pids() : parts = {parts}")
        if len(parts) >= 3:
            # Get the target task from the -3 extension
            task = parts[-3]
            target_tasks.add(task)
    target_tasks = sorted(target_tasks)
    return target_tasks

def add_stat_to_df(trial_folder, numTasksWrite, numNodes, 
                   monitor_timer_stat_io, operation, fname, task_name, task_pid, store_code):

    fname = fname.replace(".local", ".")
    # fileName removed the last 4 extensions
    fileName = ".".join(fname.split(".")[:-4])
    # fileName keep only the basename
    fileName = os.path.basename(fileName)
                        
    # Get write statistics
    # print(f"monitor_timer_stat_write = {monitor_timer_stat_write}")
    tmp_write_stat = {}
    tmp_write_stat['aggregateFilesizeMB'] = file_size_to_mb(monitor_timer_stat_io[2])

    tmp_write_stat['numTasks'] = numTasksWrite
    tmp_write_stat['numNodes'] = numNodes
    tmp_write_stat['tasksPerNode'] = tmp_write_stat['numTasks']/tmp_write_stat['numNodes']
    tmp_write_stat['transferSize'] = monitor_timer_stat_io[2]/monitor_timer_stat_io[1]
    tmp_write_stat['operation'] = operation
    tmp_write_stat['totalTime'] = monitor_timer_stat_io[0]
    tmp_write_stat['bwMiB'] = file_size_to_mb(monitor_timer_stat_io[2]/monitor_timer_stat_io[0])
    tmp_write_stat['storageType'] = store_code
    tmp_write_stat['opCount'] = monitor_timer_stat_io[1]
    tmp_write_stat['taskName'] = task_name
    tmp_write_stat['taskPID'] = task_pid
    tmp_write_stat['fileName'] = fileName
    # if w_fname == "":
    #     print(f"Write file not found for task_pid[{task_pid}] but has write stat [{tmp_write_stat}]")
    
    op = "w"
    if operation == 1: op = "r"

    # find the w_blk_trace_jsons files with the current task_pid
    w_blk_trace_jsons = glob.glob(f"{trial_folder}/*.{task_pid}.{op}_blk_trace.json")
    write_pattern = 0 # 0: seq, 1: rand        
    for w_blk_trace_json in w_blk_trace_jsons:
        with open(w_blk_trace_json) as f:
            w_blk_trace_data = json.load(f)
            # print(w_blk_trace_data)
            blk_list = w_blk_trace_data['io_blk_range']
            if blk_list[2] == -2:
                # FIXME: For now only single read and write
                write_pattern = 1
                break
    tmp_write_stat['randomOffset'] = write_pattern
    
    return tmp_write_stat
    


# TODO: find task PID's input and output to match script name
def get_wf_result_df(tests, wf_params, target_tasks,
                     numTasksWrite=1, numTasksRead=1, numNodes=1, storageType="localssd"):

    wf_df = pd.DataFrame(columns=wf_params)

    # Find folders ending with [t1, t2, t3] in the test_folders
    test_folders = glob.glob(f"{tests}/*")
    wf_trial_folders = [folder for folder in test_folders if folder.endswith("t1") or folder.endswith("t2") or folder.endswith("t3")]
    print(f"Trial folders: {wf_trial_folders} ")

    store_code = transform_store_code(storageType)
    
    for trial_folder in wf_trial_folders:
        # Find all json files in the trial folder
        # wf_json_files = glob.glob(f"{trial_folder}/*.json")


        datalife_jsons = glob.glob(f"{trial_folder}/*.datalife.json")
        blk_files = glob.glob(f"{trial_folder}/*_blk_trace.json")
        
        # print(f"blk_files = {blk_files}")
        # print(f"datalife_jsons = {datalife_jsons}")


        # Convert target_tasks to a sorted list if needed
        target_tasks = get_stat_file_pids(blk_files)
        
        # print(f"target_tasks = {target_tasks}")
        
        for datalife_json in datalife_jsons:
            task_pid = datalife_json.split("/")[-1].split(".")[1]
            # print(f"task_pid = {task_pid}")
            if task_pid in target_tasks:
                # print(f"Found taks {task_pid} in target_tasks")
                monitor_timer_stat = {}
                with open(datalife_json) as f:
                    # Get task pid from the filename monitor_timer.pid.datalife.json
                    try:
                        datalife_data = json.load(f)
                    except:
                        print(f"Error loading empty file [{f}]")

                    # Get the first key
                    task_name = list(datalife_data.keys())[0]
                    # Select 'monitor', not 'system' and 'local'
                    monitor_timer_stat = datalife_data[task_name]['monitor']
                    system_timer_stat = datalife_data[task_name]['system']

                # print(monitor_timer_stat)
 
                monitor_timer_targets = ["read", "write"] # , "close" , "seek", "stat"
                
                blk_trace_fname = [f for f in blk_files if f".{task_pid}." in f]
                # print(f"blk_trace_fname = {blk_trace_fname}")
                for fname in blk_trace_fname:
                    # Check if the file starts with r_ or w_
                    if ".r_blk_trace." in fname:
                        # Collect all statistics for read
                        read_time = 0
                        for k, v_list in monitor_timer_stat.items():
                            # read_time+= v_list[0]
                            if k in monitor_timer_targets:
                                read_time += v_list[0]
                        # Get read statistics
                        monitor_timer_stat_read = monitor_timer_stat['read']
                        # Update read time
                        monitor_timer_stat_read[0] = read_time
                        if file_size_to_mb(monitor_timer_stat_read[2]) == 0:
                            print(f"No read stat for task_name[{task_name}] task_pid[{task_pid}]")
                        else:
                        
                            tmp_read_stat = add_stat_to_df(trial_folder, numTasksRead, numNodes,
                                            monitor_timer_stat_read, 1, fname, task_name, task_pid, store_code)
                            wf_df = wf_df._append(tmp_read_stat, ignore_index=True)
                        
                    elif ".w_blk_trace." in fname:
                        # Collect all statistics for read
                        write_time = 0
                        for k, v_list in monitor_timer_stat.items():
                            # write_time+= v_list[0]
                            if k in monitor_timer_targets:
                                write_time += v_list[0]
                        # Get write statistics                        
                        monitor_timer_stat_write = monitor_timer_stat['write'] # list with [time_sec, op_cnt, io_size]
                        # Update write time
                        monitor_timer_stat_write[0] = write_time
                        if file_size_to_mb(monitor_timer_stat_write[2]) == 0:
                            print(f"No write stat for task_name[{task_name}] task_pid[{task_pid}]")
                        else:
                            monitor_timer_stat_write = monitor_timer_stat['write'] # list with [time_sec, op_cnt, io_size]
                            tmp_write_stat = add_stat_to_df(trial_folder, numTasksWrite, numNodes, 
                                        monitor_timer_stat_write, 0, fname, task_name, task_pid, store_code)
                            wf_df = wf_df._append(tmp_write_stat, ignore_index=True)
                            
                # print(f"r_fname = {r_fname}, w_fname = {w_fname}")


    # # encode string to numbers
    # le = preprocessing.LabelEncoder()
    # for col in wf_df.columns:
    #     if pd.api.types.is_string_dtype(wf_df[col]):
    #         wf_df[col] = le.fit_transform(wf_df[col])
    # print(wf_df.head(5))

    return wf_df

In [92]:
# Load 1kgenome data
onekg_data_path = "./1kgenome_data/fastflow_tests"


# Key Parameters
wf_params = ['operation', 'randomOffset', 'transferSize', 
            'aggregateFilesizeMB', 'numTasks', 'totalTime', 
            'numNodes', 'tasksPerNode', 'bwMiB', 'storageType',
            'opCount','taskName','taskPID', 'fileName']
target_tasks = ["python"] # omit srun from 1kgenome run

all_wf_df = pd.DataFrame(columns=wf_params)

def get_test_folder_dfs(test_folder, wf_params, target_tasks, 
                        storageType="localssd"):
    folder_dfs = pd.DataFrame(columns=wf_params)

    for tests in test_folder:
        # check of test folder starts with seq or par
        if tests.startswith("seq"):
            numTasksWrite = 1
            numTasksRead = 1
        else:
            # get the number after _ps
            num_tasks = int(tests.split("_")[-1].split("ps")[1])
            numTasksWrite = num_tasks
            numTasksRead = num_tasks

        # io_size_dfs
        wf_df = get_wf_result_df(f"{onekg_data_path}/{tests}", wf_params, target_tasks, 
                                numTasksWrite=numTasksWrite, numTasksRead=numTasksRead, 
                                storageType=storageType)
        print(wf_df.head(5))
        # Print size of df
        print(f"df shape: {wf_df.shape}")

        # corr_matrix(wf_df, storageType)
        folder_dfs = folder_dfs._append(wf_df, ignore_index=True)
    return folder_dfs

wf_pfs_df = pd.DataFrame(columns=wf_params)
# test_folders = ['par_3000_1n_pfs_ps300', 'par_6000_1n_pfs_ps300', 
#                 'par_9000_1n_pfs_ps300'] par_3000_10n_shm_ps300

test_folders = ['par_6000_2n_ssd_ps300']

wf_pfs_df = wf_pfs_df._append(get_test_folder_dfs(test_folders, 
                                        wf_params, target_tasks,
                                        storageType="pfs"), ignore_index=True)



Trial folders: ['./1kgenome_data/fastflow_tests/par_6000_2n_ssd_ps300/300_p_2n_ssd_t1'] 


/tmp/ipykernel_294905/2083781623.py:152: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_df = wf_df._append(tmp_read_stat, ignore_index=True)


  operation randomOffset  transferSize  aggregateFilesizeMB numTasks  \
0         1            0   8191.953215          2421.814294      300   
1         0            0  16018.200000             0.076381      300   
2         1            0   8191.970173          2421.795870      300   
3         0            0  14459.400000             0.068948      300   
4         0            0  16166.600000             0.077088      300   

   totalTime numNodes  tasksPerNode        bwMiB storageType opCount taskName  \
0   0.749422        1         300.0  3231.574973           1  309994  python3   
1   0.749493        1         300.0     0.101910           1       5  python3   
2   0.641246        1         300.0  3776.701104           1  309991  python3   
3   0.641338        1         300.0     0.107506           1       5  python3   
4   0.704856        1         300.0     0.109367           1       5  python3   

        taskPID                fileName  
0  259353-dc212     ALL.chr1.250000.vc

/tmp/ipykernel_294905/1529052691.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  folder_dfs = folder_dfs._append(wf_df, ignore_index=True)
/tmp/ipykernel_294905/1529052691.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_pfs_df = wf_pfs_df._append(get_test_folder_dfs(test_folders,


In [93]:
print(wf_pfs_df.head(5))
print(wf_pfs_df.shape)

  operation randomOffset  transferSize  aggregateFilesizeMB numTasks  \
0         1            0   8191.953215          2421.814294      300   
1         0            0  16018.200000             0.076381      300   
2         1            0   8191.970173          2421.795870      300   
3         0            0  14459.400000             0.068948      300   
4         0            0  16166.600000             0.077088      300   

   totalTime numNodes  tasksPerNode        bwMiB storageType opCount taskName  \
0   0.749422        1         300.0  3231.574973           1  309994  python3   
1   0.749493        1         300.0     0.101910           1       5  python3   
2   0.641246        1         300.0  3776.701104           1  309991  python3   
3   0.641338        1         300.0     0.107506           1       5  python3   
4   0.704856        1         300.0     0.109367           1       5  python3   

        taskPID                fileName  
0  259353-dc212     ALL.chr1.250000.vc

In [94]:
def match_script_name(tests):

    # Find folders ending with [t1, t2, t3] in the test_folders
    test_folders = glob.glob(f"{tests}/*")
    wf_trial_folders = [folder for folder in test_folders if folder.endswith("t1") or folder.endswith("t2") or folder.endswith("t3")]
    print(f"Trial folders: {wf_trial_folders} ")

    pid_input_output_dict = {}
    
    for trial_folder in wf_trial_folders:
        # Find all json files in the trial folder
        # wf_json_files = glob.glob(f"{trial_folder}/*.json")

        blk_files = glob.glob(f"{trial_folder}/*_blk_trace.json")
        unique_pids = get_stat_file_pids(blk_files)

        # monitor_tasks = get_stat_file_pids(datalife_jsons)
        # # Find the subset tasks pids from both target_tasks and monitor_tasks
        # common_tasks = set(target_tasks).intersection(monitor_tasks)
        # print(f"Target tasks num {len(target_tasks)}")
        # print(f"Common tasks: {common_tasks}")

        for pid in unique_pids:
            pid_input_output_dict[pid] = {"input": [], 
                                           "output": [],
                                           "taskName": ""}
            # Find the blk_trace_jsons files with the current task_pid # FIXME: .local should be removed later
            w_blk_trace_jsons = glob.glob(f"{trial_folder}/*.{pid}.local.w_blk_trace.json")
            r_blk_trace_jsons = glob.glob(f"{trial_folder}/*.{pid}.local.r_blk_trace.json")
            
            # FIXME: current replace .local with empty string
            w_blk_trace_jsons = [f.replace(".local", "") for f in w_blk_trace_jsons]
            r_blk_trace_jsons = [f.replace(".local", "") for f in r_blk_trace_jsons]

            if w_blk_trace_jsons:
                # Extract the file path and file name
                w_file_path = w_blk_trace_jsons[0]
                w_file_name_parts = w_file_path.split(".")
                # Remove the last 3 extensions
                w_file_name = '.'.join(w_file_name_parts[:-3])
                w_file_path_modified = '/'.join(w_file_path.split("/")[:-1]) + '/' + w_file_name
                # print(f"w_file_path_modified = {w_file_path_modified}")
                w_file_basename = w_file_path_modified.split("/")[-1]
                pid_input_output_dict[pid]['output'].append(w_file_basename)

            if r_blk_trace_jsons:
                # Extract the file path and file name
                r_file_path = r_blk_trace_jsons[0]
                r_file_name_parts = r_file_path.split(".")
                # Remove the last 3 extensions
                r_file_name = '.'.join(r_file_name_parts[:-3])
                # Reconstruct the path with the modified file name
                r_file_path_modified = '/'.join(r_file_path.split("/")[:-1]) + '/' + r_file_name
                # print(f"r_file_path_modified = {r_file_path_modified}")
                r_file_basename = r_file_path_modified.split("/")[-1]
                pid_input_output_dict[pid]['input'].append(r_file_basename)

        # print(pid_input_output_dict)
        return pid_input_output_dict

def get_wf_pid_script_dict(test_folder):

    all_wf_dict= {}

    for tests in test_folder:
        # check of test folder starts with seq or par
        if tests.startswith("seq"):
            numTasksWrite = 1
            numTasksRead = 1
        else:
            # get the number after _ps
            num_tasks = int(tests.split("_")[-1].split("ps")[1])
            numTasksWrite = num_tasks
            numTasksRead = num_tasks

        # io_size_dfs
        wf_dict = match_script_name(f"{onekg_data_path}/{tests}")

        # # corr_matrix(wf_df, storageType)
        all_wf_dict.update(wf_dict)
    return all_wf_dict


all_wf_dict = get_wf_pid_script_dict(test_folders)

# print(all_wf_dict)


Trial folders: ['./1kgenome_data/fastflow_tests/par_6000_2n_ssd_ps300/300_p_2n_ssd_t1'] 


In [95]:
import re


# Function to match a file path with patterns in task definitions
def matches_pattern(file_path, patterns):
    # Convert patterns from the JSON file to regex and match against file_path
    file_name = os.path.basename(file_path)
    # print(f"matches_pattern(): file_name = {file_name}")
    # file_name = "columns.txt"
    # patterns = ["columns\\.txt"]
    for pattern in patterns:
        # Compile the pattern to regex
        regex_pattern = re.compile(pattern)
        # print(f"Trying regex pattern: {regex_pattern.pattern}")
        if regex_pattern.fullmatch(file_name):
            # print(f"Match found: {file_name} matches {regex_pattern.pattern}")
            return True
        
    print(f"No match found for file_name = {file_name}")
    return False
        
def assign_task_names(tasks, task_definitions):
    for task_pid, details in tasks.items():
        input_paths = details['input']
        output_paths = details['output']
        task_name = 'unknown'

        for task, definition in task_definitions.items():
            input_patterns = definition['inputs']
            output_patterns = definition['outputs']

            # Check if any output path matches the output patterns
            output_match = any(matches_pattern(op, output_patterns) for op in output_paths)

            if output_match:
                if task_name != 'unknown':
                    task_name = task_name + ',' + task
                else:
                    task_name = task
            else:
                # Check if any input path matches the input patterns
                input_match = any(matches_pattern(ip, input_patterns) for ip in input_paths)
                if input_match:
                    if task_name != 'unknown':
                        task_name = task_name + ',' + task
                    else:
                        task_name = task

            # print(f"input_match: {input_match}, input_paths: {input_paths}, input_patterns: {input_patterns}")
            # print(f"output_match: {output_match}, output_paths: {output_paths}, output_patterns: {output_patterns}")
        tasks[task_pid]['taskName'] = task_name

        # FIXME: temporary fix on task names
        for k,v in tasks.items(): 
            # adjust the taskName
            outputs = v['output']
            inputs = v['input']
            # Check if any output has 'freq' in the name
            if any('freq' in op for op in outputs):
                v['taskName'] = 'frequency'
            else:
                
                v['taskName'] = v['taskName'].replace("frequency", "")
            
            
            
            # Check if any output has r'^chr(10|[0-9])-.*\.tar\.gz$'  in the name
            mo_pattern1 = r'^chr\d+-[A-Z]{3}\.txt\.tar\.gz$'
            mo_pattern2 = r'^chr\d+-[A-Z]{3}\.tar\.gz$'
            if any(re.fullmatch(mo_pattern1, op) or re.fullmatch(mo_pattern2, op) for op in outputs):
                v['taskName'] = 'mutation_overlap'
            else:
                # Remove mutation_overlap from taskName
                v['taskName'] = v['taskName'].replace("mutation_overlap", "")
            
            idm_pattern = r'^chr\d+n-\d+-\d+\.tar\.gz$'
            # Check if any input matches the pattern
            if any(re.fullmatch(idm_pattern, ip) for ip in inputs):
                v['taskName'] = 'individuals_merge'
            
            v['taskName'] = v['taskName'].replace(",", "")
            # print(f"[{k}]: {v}")
            
            # FIXME:
            if v['taskName'] == "":
                v['taskName'] = 'mutation_overlap'

# print(len(all_wf_dict))
print(f"all_wf_dict = {all_wf_dict}")

# Load task ordering json file
task_order_dict = {}
with open(f"{onekg_data_path}/1kg_script_order_fastflow.json") as f:
    task_order_dict = json.load(f)

print(task_order_dict)

# Fill in task names
assign_task_names(all_wf_dict, task_order_dict)


# Unique list of taskNames
taskNames = set([v['taskName'] for v in all_wf_dict.values()])
print(f"Unique taskNames: {taskNames}")

print(all_wf_dict)

all_wf_dict = {'1267-dc212': {'input': ['ALL.chr9.250000.vcf'], 'output': ['chr9n-401-601.tar.gz'], 'taskName': ''}, '1290-dc212': {'input': ['ALL.chr9.250000.vcf'], 'output': ['chr9n-2401-2601.tar.gz'], 'taskName': ''}, '1297-dc212': {'input': ['ALL.chr9.250000.vcf'], 'output': ['chr9n-1-201.tar.gz'], 'taskName': ''}, '1319-dc212': {'input': ['ALL.chr9.250000.vcf'], 'output': ['chr9n-601-801.tar.gz'], 'taskName': ''}, '1358-dc212': {'input': ['ALL.chr9.250000.vcf'], 'output': ['chr9n-2201-2401.tar.gz'], 'taskName': ''}, '1421-dc212': {'input': ['ALL.chr9.250000.vcf'], 'output': ['chr9n-2001-2201.tar.gz'], 'taskName': ''}, '1423-dc212': {'input': ['ALL.chr9.250000.vcf'], 'output': ['chr9n-801-1001.tar.gz'], 'taskName': ''}, '1425-dc212': {'input': ['ALL.chr9.250000.vcf'], 'output': ['chr9n-1601-1801.tar.gz'], 'taskName': ''}, '1427-dc212': {'input': ['ALL.chr9.250000.vcf'], 'output': ['chr9n-201-401.tar.gz'], 'taskName': ''}, '1429-dc212': {'input': ['ALL.chr9.250000.vcf'], 'output': [

In [96]:
# Update the taskName column in the DataFrame with 

# # print the unique PIDs from wf_pfs_df
# print(f"Unique taskPIDs in wf_pfs_df: {wf_pfs_df[
# ].unique()}")

# Create a mapping from taskPID to taskName
task_pid_to_name = {pid: info['taskName'] for pid, info in all_wf_dict.items()}

print(task_pid_to_name)

# Update the DataFrame
wf_pfs_df['taskName'] = wf_pfs_df['taskPID'].map(task_pid_to_name).fillna('unknown')

# Print the updated DataFrame
print(wf_pfs_df.head(5))

# Modify numTasks by mapping the "parallelism" from task_order_dict based on taskName
# Create a mapping from taskName to parallelism
task_name_to_parallelism = {task: info['parallelism'] for task, info in task_order_dict.items()}
print(task_name_to_parallelism)
wf_pfs_df['numTasks'] = wf_pfs_df['taskName'].map(task_name_to_parallelism).fillna(1)
# Update the tasksPerNode column
wf_pfs_df['tasksPerNode'] = wf_pfs_df['numTasks'] / wf_pfs_df['numNodes']




{'1267-dc212': 'individuals', '1290-dc212': 'individuals', '1297-dc212': 'individuals', '1319-dc212': 'individuals', '1358-dc212': 'individuals', '1421-dc212': 'individuals', '1423-dc212': 'individuals', '1425-dc212': 'individuals', '1427-dc212': 'individuals', '1429-dc212': 'individuals', '1431-dc212': 'individuals', '1433-dc212': 'individuals', '1435-dc212': 'individuals', '1437-dc212': 'individuals', '1439-dc212': 'individuals', '1441-dc212': 'individuals', '1443-dc212': 'individuals', '1445-dc212': 'individuals', '1447-dc212': 'individuals', '1449-dc212': 'individuals', '1451-dc212': 'individuals', '1453-dc212': 'individuals', '1455-dc212': 'individuals', '1458-dc212': 'individuals', '1460-dc212': 'individuals', '1462-dc212': 'individuals', '1468-dc212': 'individuals', '1471-dc212': 'individuals', '1473-dc212': 'individuals', '1475-dc212': 'individuals', '1857-dc212': 'unknown', '1859-dc212': 'individuals_merge', '1869-dc212': 'sifting', '2007-dc212': 'individuals_merge', '2008-dc2

In [97]:
# FIXME: remove rows with filename contaiing string "SIFT.chr*.vcf"
for chrom in range(0, 11):
    wf_pfs_df = wf_pfs_df[~wf_pfs_df['fileName'].str.contains(f"SIFT.chr{chrom}.vcf")]

# TIXME: temporary known tasks to sifting
wf_pfs_df['taskName'] = wf_pfs_df['taskName'].replace("unknown", "sifting")

# print(wf_pfs_df[wf_pfs_df['taskName'] == 'sifting']['fileName'])

# Save the updated DataFrame to a CSV file
wf_pfs_df.to_csv(f'{test_folders[0]}.csv', index=False)

# Split dataframe to different task names and save
unique_task_names = wf_pfs_df['taskName'].unique()

for task_name in unique_task_names:
    task_df = wf_pfs_df[wf_pfs_df['taskName'] == task_name]

    task_df.to_csv(f'{test_folders[0]}_{task_name}.csv', index=False)

In [98]:
# Calculate I/O time per taskName
task_io_time_total = wf_pfs_df.groupby('taskName')['totalTime'].sum()

task_io_time_adjust = {}
total_wf_io_time = 0
for task, io_time in task_io_time_total.items():
    # Adjust I/O time by parallelism
    io_time_adjusted = io_time / task_name_to_parallelism[task]
    task_io_time_adjust[task] = io_time_adjusted
    total_wf_io_time+=io_time_adjusted
    
    print(f" {task}: {io_time_adjusted} (sec)")
    
# # adjust by parallelism
# task_io_time = task_io_time / wf_pfs_df['numTasks'].max()

# print(task_io_time_adjust)
print(f"Total I/O time per workflow: {total_wf_io_time}")

 individuals: 1.8059174650733334 (sec)
 individuals_merge: 3.350961595 (sec)
 sifting: 0.4112709714 (sec)
Total I/O time per workflow: 5.568150031473333


In [99]:
# # in wf_pfs_df, show rows that includse "chr1n.tar.gz" in fileName and operation is read
# chr1n_tar_gz_reads = wf_pfs_df[(wf_pfs_df['fileName'].str.contains("chr1n.tar.gz")) & (wf_pfs_df['operation'] == 1)]

# print(chr1n_tar_gz_reads.shape)
# print(chr1n_tar_gz_reads)


idm_pattern = r'^chr\d+n-\d+-\d+\.tar\.gz$'
# find fileName with this pattern
idm_files = wf_pfs_df[wf_pfs_df['fileName'].str.contains(idm_pattern)]
print(idm_files.shape)

(600, 14)
